# 05 - Análise de Dados

## Objetivo

Utilizar a camada Gold para responder às perguntas de negócio definidas no início do projeto.

As análises serão realizadas a partir da tabela fato e das dimensões criadas na modelagem, utilizando consultas SQL e/ou PySpark.

Para cada pergunta, serão apresentados:

- consulta utilizada;
- resultado obtido;
- visualização quando aplicável;
- interpretação do resultado no contexto do problema.

## Pergunta 1 — Evolução dos acidentes entre 2021 e 2025

**Pergunta:** Como o número de acidentes, feridos e mortos evoluiu entre 2021 e 2025?

Para responder à pergunta, a tabela `fato_acidentes` foi relacionada à dimensão temporal `dim_data` por meio da chave `id_data`.

A análise foi agrupada por ano e considerou três indicadores principais:

- quantidade total de ocorrências de acidentes;
- quantidade total de pessoas feridas;
- quantidade total de pessoas mortas.

O objetivo é identificar a evolução desses indicadores ao longo do período analisado e verificar se houve crescimento, redução ou estabilidade entre os anos de 2021 e 2025.

In [0]:
from pyspark.sql import functions as F

In [0]:
fato = spark.table("workspace.default.fato_acidentes")
dim_data = spark.table("workspace.default.dim_data")

In [0]:
analise_ano = (
    fato
    .join(
        dim_data.select("id_data", "ano"),
        on="id_data",
        how="left"
    )
    .groupBy("ano")
    .agg(
        F.count("id_acidente").alias("acidentes"),
        F.sum("feridos").alias("feridos"),
        F.sum("mortos").alias("mortos")
    )
    .orderBy("ano")
)

display(analise_ano)

ano,acidentes,feridos,mortos
2021,64567,71873,5397
2022,64606,73065,5441
2023,67766,78463,5627
2024,73156,84526,6160
2025,72529,83550,6043


Databricks visualization. Run in Databricks to view.

### Interpretação dos resultados

Entre 2021 e 2024, observa-se uma tendência de crescimento no número de acidentes, pessoas feridas e mortes registradas nas rodovias federais analisadas.

O total de acidentes passou de 64.567 em 2021 para 73.156 em 2024. No mesmo período, o número de feridos aumentou de 71.873 para 84.526 e o número de mortos passou de 5.397 para 6.160.

Em 2025 ocorreu uma pequena redução nos três indicadores em relação a 2024, com 72.529 acidentes, 83.550 feridos e 6.043 mortos.

Considerando todo o período entre 2021 e 2025, houve aumento aproximado de 12,3% no número de acidentes, 16,2% no número de feridos e 12,0% no número de mortos.

Os resultados indicam crescimento dos registros ao longo da maior parte do período analisado, com pico em 2024 e leve redução em 2025. Esta análise é descritiva e, isoladamente, não permite determinar as causas das variações observadas.

## Pergunta 2 — Distribuição geográfica dos acidentes e vítimas fatais

**Pergunta:** Quais estados, municípios e rodovias concentram mais acidentes e vítimas fatais?

Para responder à pergunta, a tabela fato foi relacionada às dimensões de localização e de via.

Foram analisados:

- número de ocorrências;
- número total de mortos;
- ranking por unidade federativa;
- ranking por município;
- ranking por rodovia federal.

Os rankings de acidentes e de mortes são apresentados separadamente, pois os locais com maior número de ocorrências não necessariamente são os mesmos que concentram maior número de vítimas fatais.

In [0]:
from pyspark.sql import functions as F

fato = spark.table("workspace.default.fato_acidentes")
dim_localizacao = spark.table("workspace.default.dim_localizacao")
dim_via = spark.table("workspace.default.dim_via")

In [0]:
analise_uf = (
    fato
    .join(
        dim_localizacao.select("id_localizacao", "uf"),
        on="id_localizacao",
        how="left"
    )
    .groupBy("uf")
    .agg(
        F.count("id_acidente").alias("acidentes"),
        F.sum("mortos").alias("mortos")
    )
)

In [0]:
display(
    analise_uf
    .orderBy(F.desc("acidentes"))
    .limit(10)
)

uf,acidentes,mortos
MG,44502,3679
SC,39849,1921
PR,37055,2901
RJ,27669,1547
RS,24511,1619
SP,23005,1123
BA,18779,2795
GO,15940,1489
PE,14549,1539
ES,12083,802


In [0]:
display(
    analise_uf
    .orderBy(F.desc("mortos"))
    .limit(10)
)

uf,acidentes,mortos
MG,44502,3679
PR,37055,2901
BA,18779,2795
SC,39849,1921
RS,24511,1619
RJ,27669,1547
PE,14549,1539
GO,15940,1489
MT,11826,1244
MA,5778,1240


In [0]:
analise_municipio = (
    fato
    .join(
        dim_localizacao.select(
            "id_localizacao",
            "uf",
            "municipio"
        ),
        on="id_localizacao",
        how="left"
    )
    .groupBy("uf", "municipio")
    .agg(
        F.count("id_acidente").alias("acidentes"),
        F.sum("mortos").alias("mortos")
    )
)

In [0]:
display(
    analise_municipio
    .orderBy(F.desc("acidentes"))
    .limit(10)
)

uf,municipio,acidentes,mortos
DF,BRASILIA,4948,209
SP,GUARULHOS,3914,132
PR,CURITIBA,3774,132
SC,SAO JOSE,3629,69
RJ,DUQUE DE CAXIAS,3508,114
MG,BETIM,3084,108
SC,PALHOCA,2851,63
PE,RECIFE,2763,107
ES,SERRA,2559,85
PR,SAO JOSE DOS PINHAIS,2286,128


In [0]:
display(
    analise_municipio
    .orderBy(F.desc("mortos"))
    .limit(10)
)

uf,municipio,acidentes,mortos
DF,BRASILIA,4948,209
PR,CURITIBA,3774,132
SP,GUARULHOS,3914,132
PR,SAO JOSE DOS PINHAIS,2286,128
RO,PORTO VELHO,1893,128
SP,SAO PAULO,1660,123
MS,CAMPO GRANDE,1522,121
RJ,CAMPOS DOS GOYTACAZES,1673,119
BA,VITORIA DA CONQUISTA,1452,118
PR,CASCAVEL,1586,115


In [0]:
dim_via = spark.table("workspace.default.dim_via")

In [0]:
analise_br = (
    fato
    .join(
        dim_via.select("id_via", "br"),
        on="id_via",
        how="left"
    )
    .groupBy("br")
    .agg(
        F.count("id_acidente").alias("acidentes"),
        F.sum("mortos").alias("mortos")
    )
)

In [0]:
display(
    analise_br
    .orderBy(F.desc("acidentes"))
    .limit(10)
)

br,acidentes,mortos
101,59370,3414
116,52837,3600
40,16390,1008
381,16236,983
153,12965,1277
163,11653,1090
364,10737,896
277,10017,801
376,8791,650
262,8495,772


In [0]:
display(
    analise_br
    .orderBy(F.desc("mortos"))
    .limit(10)
)

br,acidentes,mortos
116,52837,3600
101,59370,3414
153,12965,1277
163,11653,1090
40,16390,1008
381,16236,983
316,5889,959
364,10737,896
230,7468,818
277,10017,801


### Interpretação dos resultados

A distribuição geográfica mostra diferenças importantes entre os locais com maior número de acidentes e aqueles com maior número de vítimas fatais.

#### Unidades Federativas

Minas Gerais apresentou tanto o maior número de acidentes quanto o maior número de mortes no período analisado, com **44.502 acidentes e 3.679 mortes**.

Santa Catarina aparece em segundo lugar no número de acidentes, com 39.849 ocorrências, mas ocupa apenas a quarta posição em número de mortes, com 1.921.

O Paraná apresentou 37.055 acidentes e 2.901 mortes, ocupando a terceira posição em acidentes e a segunda em mortes.

A Bahia chama atenção porque aparece apenas na sétima posição em quantidade de acidentes, com 18.779 ocorrências, mas sobe para a terceira posição em número de mortes, com 2.795 vítimas fatais.

Também é possível observar que estados como Mato Grosso e Maranhão não aparecem entre os dez primeiros em quantidade de acidentes, mas entram no ranking das dez unidades federativas com maior número de mortes.

Essas diferenças indicam que um maior volume de acidentes não implica necessariamente, na mesma proporção, um maior número de vítimas fatais.

#### Municípios

Brasília apresentou os maiores valores entre os municípios analisados, com **4.948 acidentes e 209 mortes**.

Guarulhos e Curitiba aparecem logo depois em número de acidentes, ambos com 132 mortes no período.

Entretanto, o ranking por mortes inclui municípios que não aparecem entre os dez primeiros em volume de acidentes, como Porto Velho, São Paulo, Campo Grande, Campos dos Goytacazes, Vitória da Conquista e Cascavel.

Isso reforça que a concentração de vítimas fatais apresenta uma distribuição diferente da concentração do número total de ocorrências.

#### Rodovias Federais

A BR-101 apresentou o maior número de acidentes, com **59.370 ocorrências**, seguida pela BR-116, com 52.837.

Quando o indicador analisado é o número de mortes, a ordem se inverte: a BR-116 apresentou **3.600 mortes**, enquanto a BR-101 registrou 3.414.

Também aparecem no ranking de mortes rodovias como a BR-316 e a BR-230, que não estão entre as dez rodovias com maior quantidade de acidentes.

Os resultados demonstram que os rankings absolutos de acidentes e mortes devem ser analisados separadamente.

## Pergunta 3 — Gravidade dos acidentes por período

**Pergunta:** Como a gravidade dos acidentes varia de acordo com o dia da semana, horário e fase do dia?

Para responder à pergunta, a tabela `fato_acidentes` foi relacionada à dimensão temporal `dim_data`.

A análise considera:

- quantidade de acidentes;
- quantidade de mortes;
- quantidade de acidentes com pelo menos uma vítima fatal;
- proporção de acidentes fatais.

Os resultados serão analisados segundo:

- dia da semana;
- horário da ocorrência;
- fase do dia.

A utilização da proporção de acidentes fatais permite comparar períodos com diferentes quantidades de ocorrências, evitando utilizar apenas o volume absoluto de mortes como medida de gravidade.

In [0]:
from pyspark.sql import functions as F

fato = spark.table("workspace.default.fato_acidentes")
dim_data = spark.table("workspace.default.dim_data")

In [0]:
analise_dia_semana = (
    fato
    .join(
        dim_data.select("id_data", "dia_semana"),
        on="id_data",
        how="left"
    )
    .groupBy("dia_semana")
    .agg(
        F.count("id_acidente").alias("acidentes"),
        F.sum("mortos").alias("mortos"),
        F.sum("acidente_fatal").alias("acidentes_fatais")
    )
    .withColumn(
        "percentual_acidentes_fatais",
        F.round(
            F.col("acidentes_fatais") / F.col("acidentes") * 100,
            2
        )
    )
    .orderBy(F.desc("percentual_acidentes_fatais"))
)

display(analise_dia_semana)

dia_semana,acidentes,mortos,acidentes_fatais,percentual_acidentes_fatais
domingo,56278,5739,4868,8.65
sábado,56111,5308,4538,8.09
sexta-feira,53009,4224,3607,6.8
quinta-feira,44343,3381,2977,6.71
segunda-feira,47208,3616,3108,6.58
quarta-feira,43392,3263,2810,6.48
terça-feira,42283,3137,2711,6.41


### Gravidade por dia da semana

A análise por dia da semana mostra que os maiores percentuais de acidentes fatais ocorrem no fim de semana.

O domingo apresentou a maior proporção de acidentes fatais, com **8,65%** das ocorrências contendo pelo menos uma vítima fatal. Em seguida aparece o sábado, com **8,09%**.

Nos dias úteis, os percentuais ficaram mais baixos, variando aproximadamente entre **6,41% e 6,80%**.

Além da maior proporção de acidentes fatais, domingo e sábado também apresentaram os maiores números absolutos de mortes entre os dias da semana analisados.

Esses resultados indicam uma associação entre o período da semana e a gravidade das ocorrências, com maior proporção de acidentes fatais durante o fim de semana.

In [0]:
analise_hora = (
    fato
    .join(
        dim_data.select("id_data", "hora"),
        on="id_data",
        how="left"
    )
    .groupBy("hora")
    .agg(
        F.count("id_acidente").alias("acidentes"),
        F.sum("mortos").alias("mortos"),
        F.sum("acidente_fatal").alias("acidentes_fatais")
    )
    .withColumn(
        "percentual_acidentes_fatais",
        F.round(
            F.col("acidentes_fatais") / F.col("acidentes") * 100,
            2
        )
    )
    .orderBy("hora")
)

display(analise_hora)

hora,acidentes,mortos,acidentes_fatais,percentual_acidentes_fatais
0,7318,848,739,10.1
1,6414,760,667,10.4
2,5798,788,691,11.92
3,6155,945,779,12.66
4,7520,1025,878,11.68
5,10286,1324,1126,10.95
6,15075,1305,1079,7.16
7,20987,1109,929,4.43
8,17194,875,732,4.26
9,14266,761,644,4.51


### Gravidade por horário da ocorrência

A análise por horário mostra que os períodos com maior quantidade de acidentes não são necessariamente aqueles com maior proporção de ocorrências fatais.

O maior volume de acidentes foi registrado às **18h**, com 25.569 ocorrências. Entretanto, nesse horário, aproximadamente **7,80%** dos acidentes tiveram pelo menos uma vítima fatal.

As maiores proporções de acidentes fatais foram observadas durante a madrugada. O maior percentual ocorreu às **3h**, quando **12,66%** dos acidentes registrados foram fatais. Outros horários da madrugada também apresentaram percentuais elevados, como 2h (11,92%), 4h (11,68%) e 5h (10,95%).

Entre aproximadamente 7h e 17h, os percentuais foram consideravelmente menores, permanecendo em sua maioria entre 4% e 6%.

A partir das 18h, a proporção de acidentes fatais volta a crescer, permanecendo próxima ou superior a 9% entre 19h e 23h.

Dessa forma, os dados indicam uma associação entre o horário da ocorrência e a gravidade dos acidentes. Embora o final da tarde concentre grande quantidade de ocorrências, a madrugada apresenta proporcionalmente maior frequência de acidentes com vítimas fatais.


In [0]:
analise_fase_dia = (
    fato
    .join(
        dim_data.select("id_data", "fase_dia"),
        on="id_data",
        how="left"
    )
    .groupBy("fase_dia")
    .agg(
        F.count("id_acidente").alias("acidentes"),
        F.sum("mortos").alias("mortos"),
        F.sum("acidente_fatal").alias("acidentes_fatais")
    )
    .withColumn(
        "percentual_acidentes_fatais",
        F.round(
            F.col("acidentes_fatais") / F.col("acidentes") * 100,
            2
        )
    )
    .orderBy(F.desc("percentual_acidentes_fatais"))
)

display(analise_fase_dia)

fase_dia,acidentes,mortos,acidentes_fatais,percentual_acidentes_fatais
Amanhecer,16601,2109,1780,10.72
Plena Noite,119581,13747,12006,10.04
Anoitecer,18821,1444,1267,6.73
Pleno dia,187621,11368,9566,5.1


### Gravidade por fase do dia

A análise por fase do dia reforça o padrão identificado na análise por horário.

O **Pleno dia** concentrou o maior número de acidentes, com 187.621 ocorrências. Entretanto, apresentou a menor proporção de acidentes fatais entre as quatro categorias, com **5,10%**.

Em contraste, o **Amanhecer** apresentou a maior proporção de acidentes fatais, com **10,72%**, apesar de registrar apenas 16.601 ocorrências.

A categoria **Plena Noite** também apresentou elevada proporção de acidentes fatais, com **10,04%**, além de concentrar 13.747 mortes no período analisado.

O **Anoitecer** apresentou um comportamento intermediário, com **6,73%** de acidentes fatais.

Esses resultados indicam que acidentes ocorridos em períodos de menor luminosidade, especialmente no amanhecer e durante a noite, apresentam maior proporção de ocorrências com vítimas fatais.

### Conclusão da Pergunta 3

A análise temporal indica que a gravidade dos acidentes varia de forma relevante conforme o período da ocorrência.

Os principais resultados foram:

- domingo e sábado apresentaram as maiores proporções de acidentes fatais entre os dias da semana;
- a madrugada apresentou os maiores percentuais de ocorrências fatais por horário, com destaque para 3h;
- o amanhecer e a plena noite apresentaram proporções de acidentes fatais aproximadamente duas vezes superiores às observadas durante o pleno dia.

Os resultados mostram que analisar apenas a quantidade de acidentes não é suficiente para avaliar gravidade.

Períodos com menor volume de ocorrências podem apresentar maior proporção de acidentes fatais, o que reforça a importância da utilização de indicadores relativos, além das contagens absolutas.

## Pergunta 4 — Condições meteorológicas e características da via

**Pergunta:** Como condições meteorológicas, tipo de pista e características do traçado da via estão associadas à gravidade dos acidentes?

Para responder à pergunta, foram utilizadas as dimensões `dim_condicoes`, `dim_via` e `dim_tracado`, além da tabela ponte `ponte_acidente_tracado`.

A análise considera:

- quantidade de acidentes;
- quantidade de mortes;
- quantidade de acidentes fatais;
- proporção de acidentes fatais.

Os resultados são avaliados separadamente para:

- condição meteorológica;
- tipo de pista;
- característica do traçado da via.

A proporção de acidentes fatais é utilizada para permitir comparação entre categorias com diferentes volumes de ocorrências.

In [0]:
dim_condicoes = spark.table("workspace.default.dim_condicoes")

In [0]:
analise_clima = (
    fato
    .join(
        dim_condicoes.select(
            "id_condicoes",
            "condicao_metereologica"
        ),
        on="id_condicoes",
        how="left"
    )
    .groupBy("condicao_metereologica")
    .agg(
        F.count("id_acidente").alias("acidentes"),
        F.sum("mortos").alias("mortos"),
        F.sum("acidente_fatal").alias("acidentes_fatais")
    )
    .withColumn(
        "percentual_acidentes_fatais",
        F.round(
            F.col("acidentes_fatais") / F.col("acidentes") * 100,
            2
        )
    )
    .orderBy(F.desc("percentual_acidentes_fatais"))
)

display(analise_clima)

condicao_metereologica,acidentes,mortos,acidentes_fatais,percentual_acidentes_fatais
Nevoeiro/Neblina,2838,392,332,11.7
null,4492,515,449,10.0
Céu Claro,214012,18452,15926,7.44
Vento,593,46,44,7.42
Nublado,54014,4427,3831,7.09
Chuva,34396,2667,2206,6.41
Garoa/Chuvisco,12118,846,697,5.75
Sol,20142,1323,1134,5.63
Neve,8,0,0,0.0
Granizo,11,0,0,0.0


In [0]:
analise_tipo_pista = (
    fato
    .join(
        dim_via.select(
            "id_via",
            "tipo_pista"
        ),
        on="id_via",
        how="left"
    )
    .groupBy("tipo_pista")
    .agg(
        F.count("id_acidente").alias("acidentes"),
        F.sum("mortos").alias("mortos"),
        F.sum("acidente_fatal").alias("acidentes_fatais")
    )
    .withColumn(
        "percentual_acidentes_fatais",
        F.round(
            F.col("acidentes_fatais") / F.col("acidentes") * 100,
            2
        )
    )
    .orderBy(F.desc("percentual_acidentes_fatais"))
)

display(analise_tipo_pista)

tipo_pista,acidentes,mortos,acidentes_fatais,percentual_acidentes_fatais
Simples,167198,19854,16452,9.84
Dupla,143585,7421,6852,4.77
Múltipla,31841,1393,1315,4.13


In [0]:
dim_tracado = spark.table("workspace.default.dim_tracado")
ponte_tracado = spark.table("workspace.default.ponte_acidente_tracado")

In [0]:
analise_tracado = (
    ponte_tracado
    .join(
        fato.select(
            "id_acidente",
            "mortos",
            "acidente_fatal"
        ),
        on="id_acidente",
        how="left"
    )
    .join(
        dim_tracado,
        on="id_tracado",
        how="left"
    )
    .groupBy("caracteristica_tracado")
    .agg(
        F.count("id_acidente").alias("acidentes"),
        F.sum("mortos").alias("mortos"),
        F.sum("acidente_fatal").alias("acidentes_fatais")
    )
    .withColumn(
        "percentual_acidentes_fatais",
        F.round(
            F.col("acidentes_fatais") / F.col("acidentes") * 100,
            2
        )
    )
    .orderBy(F.desc("percentual_acidentes_fatais"))
)

display(analise_tracado)

caracteristica_tracado,acidentes,mortos,acidentes_fatais,percentual_acidentes_fatais
Ponte,3540,455,364,10.28
Declive,32735,3959,3212,9.81
Aclive,25085,2652,2231,8.89
Curva,62755,6116,5094,8.12
Reta,241869,20669,17900,7.4
Em Obras,6656,512,424,6.37
Desvio Temporário,1467,101,79,5.39
Túnel,190,11,9,4.74
Viaduto,5210,224,206,3.95
Interseção de Vias,22802,911,804,3.53


### Observação sobre o traçado da via

O campo de traçado da via é multivalorado, ou seja, uma mesma ocorrência pode estar associada a mais de uma característica.

Por esse motivo, um acidente pode aparecer em mais de uma categoria de traçado.

Assim, os resultados dessa análise devem ser interpretados individualmente por característica, e não pela soma das categorias.

### Interpretação dos resultados

#### Condição meteorológica

A condição com maior proporção de acidentes fatais foi **Nevoeiro/Neblina**, com **11,70%** das ocorrências apresentando pelo menos uma vítima fatal.

Em seguida aparece a categoria de valor ausente (`NULL`), com 10,00%. Como essa categoria representa registros cuja condição meteorológica não estava informada, ela não deve ser interpretada como uma condição ambiental.

Entre as condições meteorológicas efetivamente identificadas, `Céu Claro` apresentou 7,44% de acidentes fatais, enquanto `Nublado` apresentou 7,09%, `Chuva` 6,41%, `Garoa/Chuvisco` 5,75% e `Sol` 5,63%.

As categorias `Neve` e `Granizo` apresentaram poucos registros, respectivamente 8 e 11 acidentes, e nenhum acidente fatal. Devido ao baixo volume de ocorrências, esses resultados devem ser interpretados com cautela.

Os dados indicam associação entre condição meteorológica e gravidade das ocorrências, com destaque para nevoeiro/neblina, que apresentou a maior proporção de acidentes fatais entre as categorias com volume relevante de registros.

#### Tipo de pista

A diferença entre os tipos de pista foi expressiva.

As pistas **simples** apresentaram 167.198 acidentes e 19.854 mortes, com **9,84%** das ocorrências classificadas como fatais.

Nas pistas **duplas**, a proporção de acidentes fatais foi de **4,77%**, enquanto nas pistas **múltiplas** foi de **4,13%**.

Assim, no conjunto analisado, acidentes registrados em pistas simples apresentaram proporção de ocorrências fatais aproximadamente duas vezes maior do que em pistas duplas ou múltiplas.

Esse resultado representa uma associação observada nos dados e não permite concluir que o tipo de pista, isoladamente, seja responsável pela maior gravidade.

#### Características do traçado

Entre as características de traçado, `Ponte` apresentou a maior proporção de acidentes fatais, com **10,28%**, seguida por `Declive`, com **9,81%**, e `Aclive`, com **8,89%**.

As ocorrências associadas a `Curva` apresentaram proporção de acidentes fatais de **8,12%**, enquanto as classificadas como `Reta` apresentaram **7,40%**.

As menores proporções foram observadas em `Rotatória` (2,26%), `Retorno Regulamentado` (3,21%) e `Interseção de Vias` (3,53%).

É importante considerar que o campo de traçado é multivalorado. Uma mesma ocorrência pode estar associada a mais de uma característica, portanto as categorias não são mutuamente exclusivas e seus totais não devem ser somados.

### Conclusão da Pergunta 4

A análise mostra que a gravidade dos acidentes varia de acordo com as condições ambientais e as características da via.

Entre os principais resultados observados:

- nevoeiro/neblina apresentou a maior proporção de acidentes fatais entre as condições meteorológicas com volume relevante;
- pistas simples apresentaram proporção de acidentes fatais significativamente superior às pistas duplas e múltiplas;
- características de traçado como ponte, declive, aclive e curva apresentaram proporções de acidentes fatais superiores às observadas em rotatórias, retornos regulamentados e interseções.

Esses resultados indicam associações entre características ambientais e rodoviárias e a gravidade das ocorrências.

Entretanto, a análise é descritiva e não permite estabelecer relações causais. Outros fatores, como velocidade praticada, fluxo de veículos, características regionais, horário e comportamento dos condutores, podem também estar relacionados aos padrões observados.

## Pergunta 5 — Causas e tipos de acidente com maior proporção de ocorrências fatais

**Pergunta:** Quais causas e tipos de acidente apresentam maior proporção de ocorrências com vítimas fatais?

Para responder à pergunta, a tabela `fato_acidentes` foi relacionada à dimensão `dim_acidente`.

A análise considera:

- quantidade total de acidentes;
- quantidade total de mortes;
- quantidade de acidentes fatais;
- proporção de acidentes fatais.

Os resultados são analisados separadamente por:

- causa do acidente;
- tipo do acidente.

A proporção de acidentes fatais é utilizada para comparar categorias com diferentes volumes de ocorrências, evitando que apenas a quantidade absoluta de acidentes determine o ranking.

In [0]:
dim_acidente = spark.table("workspace.default.dim_acidente")

In [0]:
analise_causa = (
    fato
    .join(
        dim_acidente.select(
            "id_acidente_dim",
            "causa_acidente"
        ),
        on="id_acidente_dim",
        how="left"
    )
    .groupBy("causa_acidente")
    .agg(
        F.count("id_acidente").alias("acidentes"),
        F.sum("mortos").alias("mortos"),
        F.sum("acidente_fatal").alias("acidentes_fatais")
    )
    .withColumn(
        "percentual_acidentes_fatais",
        F.round(
            F.col("acidentes_fatais") / F.col("acidentes") * 100,
            2
        )
    )
    .orderBy(F.desc("percentual_acidentes_fatais"))
)

display(analise_causa)

causa_acidente,acidentes,mortos,acidentes_fatais,percentual_acidentes_fatais
Suicídio (presumido),438,230,223,50.91
Pedestre andava na pista,3262,1395,1374,42.12
Entrada inopinada do pedestre,3326,1009,994,29.89
Transitar na contramão,11145,4163,3245,29.12
Pedestre cruzava a pista fora da faixa,2867,751,738,25.74
Participar de racha,72,16,14,19.44
Área urbana sem a presença de local apropriado para a travessia de pedestres,446,89,86,19.28
Ultrapassagem Indevida,8333,1895,1398,16.78
Pedestre - Ingestão de álcool/ substâncias psicoativas,323,53,52,16.1
Ingestão de álcool ou de substâncias psicoativas pelo pedestre,280,45,45,16.07


In [0]:
display(
    analise_causa
    .filter(F.col("acidentes") >= 100)
    .orderBy(F.desc("percentual_acidentes_fatais"))
    .limit(15)
)

causa_acidente,acidentes,mortos,acidentes_fatais,percentual_acidentes_fatais
Suicídio (presumido),438,230,223,50.91
Pedestre andava na pista,3262,1395,1374,42.12
Entrada inopinada do pedestre,3326,1009,994,29.89
Transitar na contramão,11145,4163,3245,29.12
Pedestre cruzava a pista fora da faixa,2867,751,738,25.74
Área urbana sem a presença de local apropriado para a travessia de pedestres,446,89,86,19.28
Ultrapassagem Indevida,8333,1895,1398,16.78
Pedestre - Ingestão de álcool/ substâncias psicoativas,323,53,52,16.1
Ingestão de álcool ou de substâncias psicoativas pelo pedestre,280,45,45,16.07
Iluminação deficiente,793,131,125,15.76


In [0]:
analise_tipo = (
    fato
    .join(
        dim_acidente.select(
            "id_acidente_dim",
            "tipo_acidente"
        ),
        on="id_acidente_dim",
        how="left"
    )
    .groupBy("tipo_acidente")
    .agg(
        F.count("id_acidente").alias("acidentes"),
        F.sum("mortos").alias("mortos"),
        F.sum("acidente_fatal").alias("acidentes_fatais")
    )
    .withColumn(
        "percentual_acidentes_fatais",
        F.round(
            F.col("acidentes_fatais") / F.col("acidentes") * 100,
            2
        )
    )
    .orderBy(F.desc("percentual_acidentes_fatais"))
)

display(analise_tipo)

tipo_acidente,acidentes,mortos,acidentes_fatais,percentual_acidentes_fatais
Colisão frontal,23045,8925,6749,29.29
Atropelamento de Pedestre,15313,4519,4447,29.04
Sinistro pessoal de trânsito,19,3,3,15.79
Colisão lateral sentido oposto,9582,1047,824,8.6
Eventos atípicos,1477,114,105,7.11
Atropelamento de Animal,5598,377,356,6.36
Saída de leito carroçável,52394,3446,3008,5.74
Colisão com objeto,24761,1441,1274,5.15
Colisão transversal,43190,2257,1990,4.61
Tombamento,29582,1385,1262,4.27


In [0]:
display(
    analise_tipo
    .filter(F.col("acidentes") >= 100)
    .orderBy(F.desc("percentual_acidentes_fatais"))
)

tipo_acidente,acidentes,mortos,acidentes_fatais,percentual_acidentes_fatais
Colisão frontal,23045,8925,6749,29.29
Atropelamento de Pedestre,15313,4519,4447,29.04
Colisão lateral sentido oposto,9582,1047,824,8.6
Eventos atípicos,1477,114,105,7.11
Atropelamento de Animal,5598,377,356,6.36
Saída de leito carroçável,52394,3446,3008,5.74
Colisão com objeto,24761,1441,1274,5.15
Colisão transversal,43190,2257,1990,4.61
Tombamento,29582,1385,1262,4.27
Colisão traseira,65634,3083,2743,4.18


### Interpretação dos resultados

#### Causas dos acidentes

Entre as causas com pelo menos 100 ocorrências, as maiores proporções de acidentes fatais foram observadas em situações associadas a pedestres, circulação em sentido contrário e ultrapassagens indevidas.

A categoria `Suicídio (presumido)` apresentou a maior proporção de acidentes fatais, com **50,91%**, considerando 438 ocorrências e 223 acidentes fatais.

Entre as causas diretamente relacionadas à dinâmica viária, destacam-se:

- `Pedestre andava na pista`: **42,12%**;
- `Entrada inopinada do pedestre`: **29,89%**;
- `Transitar na contramão`: **29,12%**;
- `Pedestre cruzava a pista fora da faixa`: **25,74%**;
- `Área urbana sem a presença de local apropriado para a travessia de pedestres`: **19,28%**;
- `Ultrapassagem Indevida`: **16,78%**.

Também aparecem com proporções elevadas causas relacionadas à iluminação deficiente, falta de acostamento e condições específicas envolvendo pedestres.

Em contraste, algumas causas com grande número de ocorrências apresentam proporções menores de acidentes fatais. Por exemplo, `Reação tardia ou ineficiente do condutor`, com 46.901 acidentes, apresentou 5,26% de ocorrências fatais.

Esses resultados mostram que volume de ocorrências e gravidade não são equivalentes: uma causa pode ser muito frequente e, ainda assim, apresentar menor proporção de acidentes fatais do que causas menos comuns.

#### Tipos de acidente

Entre os tipos de acidente, `Colisão frontal` apresentou a maior proporção de ocorrências fatais, com **29,29%**, seguida muito de perto por `Atropelamento de Pedestre`, com **29,04%**.

A `Colisão lateral sentido oposto` apresentou **8,60%**, enquanto os demais tipos ficaram abaixo desse valor.

Outros resultados relevantes foram:

- `Eventos atípicos`: 7,11%;
- `Atropelamento de Animal`: 6,36%;
- `Saída de leito carroçável`: 5,74%;
- `Colisão com objeto`: 5,15%;
- `Colisão transversal`: 4,61%;
- `Tombamento`: 4,27%;
- `Colisão traseira`: 4,18%.

O tipo `Incêndio` apresentou apenas **0,07%** de acidentes fatais, apesar de possuir 7.430 ocorrências.

Os resultados indicam que colisões frontais e atropelamentos de pedestres possuem associação muito mais forte com desfechos fatais do que outros tipos de ocorrência.

### Conclusão da Pergunta 5

A análise mostra que determinadas causas e tipos de acidente estão fortemente associados a maior gravidade das ocorrências.

Entre as causas, destacam-se situações envolvendo pedestres, circulação na contramão e ultrapassagens indevidas.

Entre os tipos de acidente, colisões frontais e atropelamentos de pedestres apresentaram, com ampla diferença, as maiores proporções de acidentes fatais.

Esses resultados reforçam a importância de analisar não apenas o número de ocorrências, mas também a proporção de acidentes fatais em cada categoria.

A análise é descritiva e não permite concluir que uma determinada causa ou tipo de acidente seja, isoladamente, responsável pelo desfecho fatal.

## Conclusão Geral das Perguntas de Negócio

A análise dos acidentes registrados pela Polícia Rodoviária Federal entre 2021 e 2025 permitiu identificar padrões temporais, geográficos, ambientais e rodoviários associados à ocorrência e à gravidade dos acidentes nas rodovias federais brasileiras.

Em relação à evolução temporal, os dados mostraram crescimento no número de acidentes, feridos e mortos entre 2021 e 2024, seguido por uma pequena redução em 2025. O ano de 2024 apresentou os maiores valores dos três indicadores no período analisado.

Na distribuição geográfica, observou-se que os locais com maior número de acidentes não são necessariamente aqueles com maior número de mortes. Minas Gerais apresentou os maiores valores absolutos entre as unidades federativas, enquanto estados como Bahia, Mato Grosso e Maranhão ganharam maior relevância quando o indicador considerado foi o número de vítimas fatais. O mesmo comportamento foi observado em municípios e rodovias federais, reforçando a importância de analisar volume de ocorrências e gravidade separadamente.

A análise temporal da gravidade também revelou diferenças relevantes. Sábados e domingos apresentaram as maiores proporções de acidentes fatais entre os dias da semana. Por horário, a madrugada concentrou os maiores percentuais de ocorrências fatais, com destaque para as 3h. A análise por fase do dia confirmou esse padrão, com o amanhecer e a plena noite apresentando proporções de acidentes fatais significativamente superiores às observadas durante o pleno dia.

As condições ambientais e características da via também apresentaram associação com a gravidade. Entre as condições meteorológicas, nevoeiro e neblina apresentaram a maior proporção de acidentes fatais entre as categorias com volume relevante de registros. Pistas simples apresentaram proporção de acidentes fatais aproximadamente duas vezes superior às pistas duplas e múltiplas. Entre as características de traçado, ponte, declive, aclive e curva apresentaram percentuais superiores aos observados em rotatórias, retornos regulamentados e interseções.

Por fim, a análise das causas e tipos de acidente mostrou que determinadas categorias estão associadas a proporções muito mais elevadas de ocorrências fatais. Situações envolvendo pedestres, circulação na contramão e ultrapassagens indevidas apareceram entre as causas mais críticas. Entre os tipos de acidente, colisões frontais e atropelamentos de pedestres apresentaram, com ampla diferença, as maiores proporções de acidentes fatais.

De forma geral, os resultados mostram que a gravidade dos acidentes não depende apenas da quantidade de ocorrências registradas. Fatores relacionados ao período da ocorrência, localização, condições ambientais, características da via e natureza do acidente apresentam padrões distintos de gravidade.

## Autoavaliação

O objetivo principal do projeto foi construir um pipeline de Engenharia de Dados utilizando dados abertos da Polícia Rodoviária Federal, desde a ingestão dos arquivos brutos até a criação de uma camada analítica estruturada para responder às perguntas de negócio definidas no início do trabalho.

Considero que o objetivo foi atingido.

O pipeline foi desenvolvido no Databricks utilizando PySpark, Delta Lake e Unity Catalog, seguindo uma organização em camadas Bronze, Silver e Gold.

Na camada Bronze, os arquivos referentes aos anos de 2021 a 2025 foram consolidados e preservados com campos de rastreabilidade.

Na etapa de qualidade, foram realizadas verificações de completude, unicidade, consistência, acurácia e valores extremos. Essa análise permitiu identificar situações como marcadores textuais de ausência, diferenças de padronização entre categorias e campos multivalorados.

Na camada Silver, foram realizadas conversões de tipos, padronização de valores, tratamento de ausências e criação de atributos derivados.

Na camada Gold, foi desenvolvida uma modelagem dimensional composta por uma tabela fato, dimensões e uma tabela ponte para representar corretamente o campo multivalorado de traçado da via.

As cinco perguntas de negócio definidas inicialmente puderam ser respondidas a partir da estrutura criada.

Entre os principais aprendizados do trabalho está a importância de compreender o objetivo analítico antes de construir o pipeline, pois as perguntas de negócio influenciaram diretamente a seleção dos campos, as transformações realizadas e a modelagem das tabelas.

Também foi possível perceber que problemas de qualidade nem sempre aparecem como valores nulos ou erros técnicos. Em diversos casos, foi necessário interpretar semanticamente os dados para identificar situações como `IGNORADO`, `N/A` e categorias inconsistentes.

Uma das principais dificuldades foi a modelagem do campo `tracado_via`, que permite múltiplas características para a mesma ocorrência. Para evitar perda de informação ou duplicação da tabela fato, foi utilizada uma dimensão específica associada a uma tabela ponte.

Outra dificuldade ocorreu na criação dos relacionamentos da tabela fato com dimensões contendo valores nulos. O problema foi resolvido utilizando comparações compatíveis com valores nulos durante os joins.

Como limitação, o trabalho utiliza dados observacionais e realiza análises descritivas. Portanto, os resultados identificam associações entre características das ocorrências e sua gravidade, mas não permitem estabelecer relações causais.

Além disso, não foram utilizadas variáveis externas como fluxo de veículos, extensão das rodovias, volume de tráfego ou condições socioeconômicas. Dessa forma, rankings baseados em números absolutos não devem ser interpretados diretamente como medidas de risco.

Como evolução futura, o pipeline poderia ser automatizado para ingestão de novos arquivos, incluir controles adicionais de qualidade, incorporar novas fontes de dados e disponibilizar os resultados por meio de dashboards analíticos.

De forma geral, o desenvolvimento do MVP permitiu aplicar na prática conceitos de ingestão, tratamento, qualidade, modelagem, linhagem e análise de dados, demonstrando como um pipeline de Engenharia de Dados pode transformar arquivos brutos em uma estrutura organizada e adequada para consumo analítico.